In [ ]:
# Load packages
from pathlib import Path
import os
import re
import sys
import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


from kardemumma import *
from kardemumma.proteomedge import fetch_qreps_table

In [ ]:

# Skyline data path
skyline_path = "data/MARTHA/DE17501_Martha_results_dotp.csv"

# Import skyline data (includes column "Isotope Label Type" from Precursor)
skyline_importer = ImportFile(skyline_path)
skyline_data = skyline_importer.import_skyline_file()

In [ ]:
# Isotope Label Type is added in ImportFile.import_skyline_file() (from Precursor).

# qREPs spike levels
qREPs_spike_levels = pd.read_csv('ratio/DE17501_ratio.csv')

# SDRF
sdrf_path = 'sdrf/sdrf_MARTHA_combined.sdrf.tsv'
sdrf_data = import_sdrf_file(sdrf_path)


In [ ]:
cross_check_skyline_sdrf(skyline_df = skyline_data, sdrf_df = sdrf_data, match_mode="exact")

In [ ]:
# Import and use function from output_test.py to get iRT peptides
iRT_peptides = get_irt_peptides(skyline_data)

print('This is the iRT peptides:')
print(iRT_peptides)

In [ ]:
# Checking possible QC samples from Skyline results

qc_samples = skyline_importer.suggest_qc_samples(skyline_data)
skyline_checker = CheckSkylineFile(skyline_path)

# Suggest qc samples
print('This is the qc samples:')
print(f'Number of qc samples: {len(qc_samples)}') 
print(qc_samples)


In [ ]:
# Get test samples (exclude QC samples from all replicate names)
skyline_checker = CheckSkylineFile(skyline_path)
test_samples = skyline_checker.get_test_samples(qc_samples, skyline_data)
test_data = skyline_checker.get_test_data(test_samples, skyline_data)
print('This is the test samples:')
print(f'Number of test samples: {len(test_samples)}')

In [ ]:
# Remove all rows in skyline_data that contains any of the qc_replicates
skyline_data = test_data
print(test_data.shape)

In [ ]:
plot_library_dot_product_distribution(skyline_data)

In [ ]:
skyline_clean = filter_library_dot_product(skyline_data, threshold=0.6)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = summarise_peptide_counts(skyline_clean)


In [ ]:
report_summary, peptide_list = report_peptide_protein_summary(peptide_counts)

In [ ]:
plot_heavy_light_scatter(peptide_counts)

In [ ]:
filtered_peptide_counts = filter_peptide_counts(peptide_counts, light_cutoff=700, heavy_cutoff=700)

filtered_peptide_counts.head()


In [ ]:
selected_peptides_report,selected_peptides = report_peptide_protein_summary(filtered_peptide_counts)

In [ ]:
from kardemumma.importer import MergeFiles

skyline_merge = MergeFiles(skyline_data, sdrf_data, selected_peptides).merge_files()
skyline_pool = MergeFiles(skyline_data, sdrf_data, selected_peptides).select_pool_data(skyline_merge, col_sample='characteristics[Sample]', sample_value='Pool')

In [ ]:
skyline_pool.head()

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
plot_pool_boxplot(skyline_pool)


In [ ]:
skyline_pool.head()

In [ ]:
# Calculate intra-plate CV

peptide_plate_stats = calculate_intra_plate_cv(skyline_pool, col_name='characteristics[Plate]')
plot_intra_plate_cv_stats(peptide_plate_stats, col_name='characteristics[Plate]')

In [ ]:
interplate_cv = calculate_inter_plate_cv(peptide_plate_stats)
plot_inter_plate_cv_kde(interplate_cv)

In [ ]:
interplate_cv.head()

In [ ]:
# Print interplate_cv froom low to high
print('This is the interplate_cv from low to high:')
print(interplate_cv[interplate_cv['inter_plate_cv'] < 0.1])

In [ ]:
plot_cumulative_peptide_count_by_cv(interplate_cv)

In [ ]:
selected_peptides = get_lowest_cv_peptides(interplate_cv, 10)
selected_peptides

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 
# using only peptides in the top 10% lowest inter-plate CV for normalization

pool_selected_df = skyline_pool[skyline_pool['Peptide Sequence'].isin(selected_peptides)]
# Filter pool_data to include only those peptides for normalization calculation
pool_selected_df

In [ ]:
plate_factor_table, conversion_factors, model = get_plate_conversion_factors(pool_selected_df, col_plate='characteristics[Plate]', log_transform=True  )

In [ ]:
plate_factor_table

In [ ]:
conversion_factors

In [ ]:


# Use the adjust_ratio_by_plate function above to add RatioLightToHeavy_adj to skyline_merge
# First, construct a Plate column of the correct type for the function
skyline_merge_adj = skyline_merge.copy()
# Make sure 'Plate' column exists and matches conversion_factors keys
skyline_merge_adj['Plate'] = skyline_merge_adj['characteristics[Plate]']
skyline_merge_adj = adjust_ratio_by_plate(skyline_merge_adj, conversion_factors)


In [ ]:
# From skyline_merge_adj, filter to pool samples
pool_data_adj = skyline_merge_adj[skyline_merge_adj['characteristics[Sample]'] == 'Pool'].copy()

pool_data_adj.head()


In [ ]:
conversion_factors

In [ ]:
plot_pool_boxplot(pool_data_adj)

In [ ]:
plot_pool_boxplot(pool_data_adj, col_ratio='RatioLightToHeavy_adj')

# Calculate absolute quantification

In [ ]:

# Add new column call qRePS which is the string before _ in Protein Name col
skyline_merge_adj['qRePS'] = skyline_merge_adj['Protein Name'].str.split('_').str[0]

In [ ]:
# Merge qREPs_spike_levels, to skyline_merge_adj by qRePs
skyline_merge_adj = pd.merge(skyline_merge_adj, qREPs_spike_levels, on=['qRePS'], how='left')
# Add new column call qRePs which is the string before _ in Protein Name col

# Add new column called Protein conc [pmol] which is equal to RatioLightToHeavy_adj * Amount Per well [pmol]
skyline_merge_adj['Protein conc [pmol]'] = skyline_merge_adj['RatioLightToHeavy_adj'] * skyline_merge_adj['Amount per well [pmol]']

# Round Protein conc [pmol] to 4 decimal places
skyline_merge_adj['Protein conc [pmol]'] = skyline_merge_adj['Protein conc [pmol]'].round(4)


In [ ]:
skyline_merge_adj.head()

In [ ]:
# Export wide format for qREPs


# Select columns to export
export_cols = ['Replicate', 'Peptide Sequence', 'Protein Name', 'qRePS', 'Protein conc [pmol]']

# Pivot the dataframe to wide format
skyline_merge_adj_wide = skyline_merge_adj.pivot(
    index=['qRePS', 'Peptide Sequence', 'Protein Name'],
    columns='Replicate',
    values='Protein conc [pmol]'
)

# Export to csv status
# skyline_merge_adj_wide.to_csv('export/MARTHA_conc_normalized.csv', index=True)


